In [1]:
!pip install -q groq fpdf

In [2]:
!pip install streamlit

In [6]:
!pip install langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.8 MB/s eta 0:00:00
  Attempting uninstall: groq
    Found existing installation: groq 1.2.0
    Uninstalling groq-1.2.0:
      Successfully uninstalled groq-1.2.0


In [12]:
%%writefile app.py
import streamlit as st
from groq import Groq
from fpdf import FPDF
from google.colab import userdata
import os

# 1. Setup the Groq Client
try:
    client = Groq(api_key=userdata.get('RealEst'))
except Exception as e:
    st.error(f"API Key issue: {e}")

st.set_page_config(page_title="Groq Resume Architect", page_icon="⚡")

st.title("⚡ Groq AI Resume Architect")
st.markdown("Generate a professional resume in seconds using the world's fastest AI inference.")

# 2. Input Form
with st.form("resume_form"):
    col1, col2 = st.columns(2)
    with col1:
        name = st.text_input("Full Name", placeholder="John Doe")
        email = st.text_input("Email")
        major = st.text_input("Major / Education")
    with col2:
        phone = st.text_input("Phone Number")
        linkedin = st.text_input("LinkedIn URL")
        job_title = st.text_input("Target Job Title", placeholder="Data Scientist")

    experience = st.text_area("Experience", placeholder="List your key roles and achievements...")
    skills = st.text_area("Skills", placeholder="e.g., Python, SQL, Project Management")

    submitted = st.form_submit_button("Generate Resume Now")

# 3. Groq Inference
if submitted:
    if not name or not job_title:
        st.warning("Please enter at least your Name and Target Job Title.")
    else:
        with st.spinner("Groq is processing at lightning speed..."):
            prompt = f"""
            System: You are a professional executive resume writer.
            User: Create a clean, modern, and high-impact resume based on these details:
            Name: {name}
            Target Job: {job_title}
            Education: {major}
            Skills: {skills}
            Experience: {experience}
            Contact: {email} | {phone} | {linkedin}

            Instructions: Use professional action verbs. Format with clear headings.
            Output only the resume text.
            """

            # Calling the Groq API
            completion = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.7,
            )

            resume_text = completion.choices[0].message.content

            st.subheader("Preview")
            st.text_area("Edit text here if needed:", value=resume_text, height=400)

            # 4. PDF Generation Logic
            pdf = FPDF()
            pdf.add_page()
            pdf.set_font("Arial", size=11)

            # Encoding fix for PDF
            clean_text = resume_text.encode('latin-1', 'ignore').decode('latin-1')

            for line in clean_text.split('\n'):
                pdf.multi_cell(0, 8, txt=line)

            pdf_path = "resume.pdf"
            pdf.output(pdf_path)

            with open(pdf_path, "rb") as f:
                st.download_button(
                    label="📥 Download Resume PDF",
                    data=f,
                    file_name=f"{name.replace(' ', '_')}_Resume.pdf",
                    mime="application/pdf"
                )

Overwriting app.py


In [10]:
!pip install pyngrok

In [11]:
import os
from google.colab import userdata
from pyngrok import ngrok
import subprocess

# 1. Pull the secret key securely from the notebook's environment
# Change 'NGROK_AUTH' to whatever you named your secret
try:
    NGROK_TOKEN = userdata.get('NGROK_AUTH')
    ngrok.set_auth_token(NGROK_TOKEN)
    print("✅ Secret key loaded successfully!")
except Exception as e:
    print(f"❌ Could not find secret: {e}")

# 2. Clean up any old sessions
ngrok.kill()
os.system("pkill streamlit")

# 3. Launch the App
try:
    # Run Streamlit in the background
    subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])

    # Create the public tunnel
    public_url = ngrok.connect(8501).public_url
    print(f"\n🚀 YOUR APP IS LIVE!")
    print(f"Click here to view: {public_url}")
except Exception as e:
    print(f"❌ Connection Failed: {e}")

✅ Secret key loaded successfully!

🚀 YOUR APP IS LIVE!
Click here to view: https://cocciferous-dioicous-gino.ngrok-free.dev
